In [13]:
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import LogisticRegression
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures,MinMaxScaler,StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
import seaborn as sns


df=pd.read_csv('boston.csv')
x_train, x_test, y_train, y_test = train_test_split(df[['LSTAT', 'RM', 'PTRATIO', 'INDUS', 'TAX']], df['MEDV'], test_size=0.2, random_state=42)

In [11]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def sigmoid_derivative(z):
    s = sigmoid(z)
    return s * (1 - s)

def softmax(z):
    exp_z = np.exp(z - np.max(z, axis=1, keepdims=True))
    return exp_z / np.sum(exp_z, axis=1, keepdims=True)
def ann_numpy_regressor(X_train, y_train, X_test, y_test, epochs=200, hidden_size=16, learning_rate=0.01):
    np.random.seed(42)
    input_size = X_train.shape[1]
    output_size = 1

    W1 = np.random.randn(input_size, hidden_size) * 0.01
    b1 = np.zeros((1, hidden_size))
    W2 = np.random.randn(hidden_size, output_size) * 0.01
    b2 = np.zeros((1, output_size))

    for epoch in range(epochs):
        # Forward
        Z1 = np.dot(X_train, W1) + b1
        A1 = sigmoid(Z1)
        Z2 = np.dot(A1, W2) + b2  # Linear output for regression

        loss = np.mean((Z2 - y_train.reshape(-1, 1))**2)  # MSE

        # Backward
        dZ2 = (Z2 - y_train.reshape(-1, 1))
        dW2 = np.dot(A1.T, dZ2)
        db2 = np.sum(dZ2, axis=0, keepdims=True)

        dA1 = np.dot(dZ2, W2.T)
        dZ1 = dA1 * sigmoid_derivative(Z1)
        dW1 = np.dot(X_train.T, dZ1)
        db1 = np.sum(dZ1, axis=0, keepdims=True)

        # Update
        W1 -= learning_rate * dW1
        b1 -= learning_rate * db1
        W2 -= learning_rate * dW2
        b2 -= learning_rate * db2

        if epoch % 50 == 0:
            print(f"Epoch {epoch}, MSE: {loss:.4f}")

    # Predict
    Z1_test = np.dot(X_test, W1) + b1
    A1_test = sigmoid(Z1_test)
    y_pred = np.dot(A1_test, W2) + b2

    test_mse = np.mean((y_pred - y_test.reshape(-1, 1))**2)
    print(f"\nTest MSE: {test_mse:.4f}")
    return y_pred


In [12]:
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

# Call function
ann_numpy_regressor(x_train_scaled, y_train.values, x_test_scaled, y_test.values)


Epoch 0, MSE: 606.7437
Epoch 50, MSE: 64839111900708089967646970337640675863973789767303168.0000
Epoch 100, MSE: 125659526402995723299279987007152889889698715573058353085150345908648881188461135005036568424354938880.0000
Epoch 150, MSE: 243530734967588264223513450178623493495550163591011737334992277550991972902365507715345944756500517212697887768009296446063437852737719969446967115776.0000

Test MSE: 471967550503515566051480685301700379584534285582399887089999669432271642309780186626669258566990406913533270960654144757333475074568853170104258875353825062830868512533208142213111797782775582425088.0000


C:\Users\Hashir\AppData\Local\Temp\ipykernel_2544\1281197834.py:2: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-z))


array([[-6.86998945e+98],
       [-6.86998945e+98],
       [-6.86998945e+98],
       [-6.86998945e+98],
       [-6.86998945e+98],
       [-6.86998945e+98],
       [-6.86998945e+98],
       [-6.86998945e+98],
       [-6.86998945e+98],
       [-6.86998945e+98],
       [-6.86998945e+98],
       [-6.86998945e+98],
       [-6.86998945e+98],
       [-6.86998945e+98],
       [-6.86998945e+98],
       [-6.86998945e+98],
       [-6.86998945e+98],
       [-6.86998945e+98],
       [-6.86998945e+98],
       [-6.86998945e+98],
       [-6.86998945e+98],
       [-6.86998945e+98],
       [-6.86998945e+98],
       [-6.86998945e+98],
       [-6.86998945e+98],
       [-6.86998945e+98],
       [-6.86998945e+98],
       [-6.86998945e+98],
       [-6.86998945e+98],
       [-6.86998945e+98],
       [-6.86998945e+98],
       [-6.86998945e+98],
       [-6.86998945e+98],
       [-6.86998945e+98],
       [-6.86998945e+98],
       [-6.86998945e+98],
       [-6.86998945e+98],
       [-6.86998945e+98],
       [-6.8

In [21]:
from tensorflow.keras.utils import to_categorical

y_encoded = to_categorical(y)  # Now shape is (n_samples, 2)

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_encoded, test_size=0.3, random_state=42, stratify=y
)


NameError: name 'y' is not defined

In [20]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
model = Sequential()
model.add(Dense(16, input_dim=x_train.shape[1], activation='relu'))  # Adjust to 6 or more features
model.add(Dense(8, activation='relu'))
model.add(Dense(2, activation='softmax'))  # Binary classification, softmax for 2 neurons

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])


history = model.fit(x_train, y_train, epochs=100, batch_size=8, validation_split=0.2, verbose=1)


Epoch 1/100


c:\Users\Hashir\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


ValueError: Arguments `target` and `output` must have the same shape. Received: target.shape=(None, 1), output.shape=(None, 2)

In [19]:
print(x_train.shape)
print(x_test.shape)
print(y_train.shape)
print(y_test.shape)


(505, 6)
(90, 6)
(505,)
(90,)
